Visualize the regression results with actual vs. predicted values by plotting interactive scatter plots with confidence intervals, color-coding data points based on error magnitude, and analyzing residual distributions for systematic biases.

In [18]:
import joblib
import numpy as np
from scipy.sparse import load_npz

In [2]:
best_bayes_model = joblib.load('best_bayes_model.pkl')
print("Best Bayesian model loaded successfully!")

Best Bayesian model loaded successfully!


In [17]:
x_trained_processed = load_npz('x_trained_processed.npz')
y_train = np.load('y_train.npy')
x_test_processed = load_npz('x_test_processed.npz')
y_test = np.load('y_test.npy')

print("Test data shape:", x_test_processed.shape)
print("Target shape:", y_test.shape)

Test data shape: (4697, 3045)
Target shape: (4697,)


In [10]:
y_pred = best_bayes_model.predict(x_test_processed)

print("Predictions generated:", len(y_pred))
print("First 10 predictions:", y_pred[:10])

Predictions generated: 4697
First 10 predictions: [4.45852958 3.86791003 4.26638705 3.45974841 4.46271318 4.71083116
 5.06008486 5.20910081 4.21511374 4.37803229]


In [11]:
predicted_ratings = np.clip(np.rint(y_pred), 1, 5).astype(int)

print("First 10 predicted ratings:", predicted_ratings[:10])

First 10 predicted ratings: [4 4 4 3 4 5 5 5 4 4]


Interactive scatter plot

In [13]:
import plotly.express as px
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [15]:
regression_res = pd.DataFrame({
    'Actual': np.asarray(y_test).ravel(),
    'Predicted': np.asarray(y_pred).ravel()
})

# Calculate prediction errors
regression_res['Residual'] = (regression_res['Actual'] - regression_res['Predicted'])

regression_res['Absolute_Error'] = (regression_res['Residual'].abs())

# Interactive scatter plot
fig = px.scatter(
    regression_res,
    x='Actual',
    y='Predicted',
    color='Absolute_Error',
    color_continuous_scale='Viridis',
    opacity=0.65,
    hover_data=['Residual', 'Absolute_Error'],
    title='Actual vs Predicted Customer Ratings',
    labels={
        "Actual": "Actual Rating",
        "Predicted": "Predicted Rating",
        "Absolute_Error": "Absolute Prediction Error"
    }
)

# Ideal prediction line
fig.add_shape(
    type='line',
    x0=1, y0=1,
    x1=5, y1=5,
    line=dict(color='red', dash='dash', width=2)
)

fig.update_layout(template='plotly_white', width=850, height=600)

fig.show()


print("MAE:", round(mean_absolute_error(y_test, y_pred), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred)), 4))
print("R² Score:", round(r2_score(y_test, y_pred), 4))


MAE: 0.5608
RMSE: 0.7428
R² Score: 0.5474


Residual distribution

In [19]:

# Residual distribution
fig = px.histogram(
    regression_res,
    x="Residual",
    nbins=50,
    marginal="box",
    opacity=0.75,
    title="Residual Distribution — Bayesian Ridge Model",
    labels={"Residual": "Actual - Predicted Rating"}
)

fig.add_vline(
    x=0,
    line_dash="dash",
    line_color="red",
    annotation_text="Zero Error"
)

fig.update_layout(
    template="plotly_white",
    width=850,
    height=500,
    yaxis_title="Number of Predictions"
)

fig.show()

print("Mean residual:",round(regression_res["Residual"].mean(), 4))

print("Median residual:",round(regression_res["Residual"].median(), 4))

print("Residual standard deviation:",round(regression_res["Residual"].std(), 4))


Mean residual: -0.0061
Median residual: 0.0708
Residual standard deviation: 0.7428


Interactive residual plot

In [21]:
fig = px.scatter(
    regression_res,
    x='Predicted',
    y='Residual',
    color='Absolute_Error',
    color_continuous_scale='Viridis',
    opacity=0.65,
    hover_data=["Actual", "Predicted", "Residual"],
    title="Residuals vs Predicted Ratings",
    labels={
        "Predicted": "Predicted Rating",
        "Residual": "Residual (Actual - Predicted)",
        "Absolute_Error": "Absolute Prediction Error"
    }
)

# Reference line for zero residual
fig.add_hline(
    y=0,
    line_dash='dash',
    line_color='red',
    annotation_text='Zero Residual'  
)

fig.update_layout(
    template='plotly_white',
    width=850,
    height=550
)

fig.show()


Prediction intervals (95%)

In [22]:
# Predict training ratings using the loaded model
y_train_pred = best_bayes_model.predict(x_trained_processed)

train_residual = np.asarray(y_train).ravel() - y_train_pred

# Approximate 95% prediction interval width
lower_error = np.percentile(train_residual, 2.5)
upper_error = np.percentile(train_residual, 97.5)

# Prepare test predictions and interval bounds
interval_df = regression_res.copy()

interval_df['Lower_Bound'] = (interval_df['Predicted'] + lower_error)
interval_df['Upper_Bound'] = (interval_df['Predicted'] + upper_error)

# Interactive plot with approximate interval bounds
fig = px.scatter(
    interval_df,
    x='Actual',
    y='Predicted',
    color='Absolute_Error',
    color_continuous_scale='Viridis',
    opacity=0.55,
    title="Actual vs Predicted Ratings with Approximate 95% Prediction Intervals",
    hover_data=["Lower_Bound", "Upper_Bound", "Absolute_Error"]
)

fig.add_shape(
    type="line",
    x0=1, y0=1,
    x1=5, y1=5,
    line=dict(color="red", dash="dash", width=2)
)

fig.update_layout(
    template="plotly_white",
    width=900,
    height=600
)

fig.show()

print("Approximate lower residual bound:", round(lower_error, 3))
print("Approximate upper residual bound:", round(upper_error, 3))

Approximate lower residual bound: -1.557
Approximate upper residual bound: 1.105


Conclusion — Regression Result Visualization and Residual Analysis
The Bayesian Ridge Regression model was evaluated using interactive visualizations of actual versus predicted customer ratings, residual distribution, and residuals versus predicted ratings. The actual-versus-predicted scatter plot used color coding based on absolute prediction error to identify observations with larger errors.

The model achieved a Mean Absolute Error (MAE) of 0.5608, Root Mean Squared Error (RMSE) of 0.7428, and R² score of 0.5474, indicating that it explains approximately 54.74% of the variation in customer ratings. The residual distribution was concentrated around zero, with a mean residual of approximately −0.0061, suggesting minimal overall average signed error. However, some predictions had larger residuals, indicating that prediction accuracy varies across observations.

The approximate residual-based 95% prediction interval used residual bounds of −1.557 to +1.105 rating points around each predicted value. These are approximate prediction bounds rather than formal statistical confidence intervals. Residual analysis helps identify prediction errors and potential systematic bias, while the visualizations provide a clearer understanding of model performance and limitations.

Overall, the visual analysis indicates that the model provides useful rating predictions, but some errors remain. Further improvements may be possible through feature engineering, model comparison, and additional validation.